# Create files mapping FACS well and plate to basic allele metadata

This script takes cherry picking and allele info files as input and produces a table relating FACS well and plate IDs to allele IDs and symbols. 

In [94]:
import pandas as pd
import numpy as np
import os

In [69]:
dualipa_inputs = "../1_inputs"
dualipa_outputs = "../3_outputs"
allele_collection_inputs = "../../../1_allele_collection/1_inputs"
allele_collection_outputs = "../../../1_allele_collection/3_outputs"

In [ ]:
# Download raw FACS data from Zenodo
bash_file = f"{dualipa_inputs}/download_raw_FACS.sh"
os.system(f"bash {bash_file}")

In [114]:
# Get allele metadata
allele_info_df = pd.read_table(f"{allele_collection_inputs}/VarChamp_Consolidated_allele_collection.tsv")
other_cols = [i for i in allele_info_df.columns if "mutation_id_" not in i]
mut_cols = [i for i in allele_info_df.columns if "mutation_id_" in i]

allele_info_df = allele_info_df.melt(id_vars=other_cols, 
                                     value_vars=mut_cols, 
                                     var_name='mut_type', 
                                     value_name='mut_id')

In [115]:
# Get gene-level metadata
gene_info_df = pd.read_csv(f"{allele_collection_outputs}/orf_info.csv")
gene_info_df = gene_info_df[['orf_id', 'symbol']]
orf_to_symbol = gene_info_df.set_index("orf_id")["symbol"].to_dict()
orf_to_symbol = {str(k): v for k, v in orf_to_symbol.items()}

Vidal lab ran experiment once and analyzed the data. Some alleles didn't pass the qa/qc thresholds or got no data. Batch5 is is a re-run for all alleles that failed in the first set of batches. Here, we process everything together from scratch. An allele should always be compared to the wt on the same batch and plate. The summary stats can then be compared across batches. 

In [116]:
# Read in cherry picking files for both the initial batches and the second round
cp_col = ['orf_id', 'mut_id', 'node_type','dest_pla_id', 'dest_pla', 'dest_well']

pdest_layout_initial_batches_df = pd.read_csv(f'{dualipa_inputs}/cherry_picking_files/summary_CP_VUSDUALXP.csv', index_col=0)[cp_col]
pdest_layout_b5_df = pd.read_csv(f'{dualipa_inputs}/cherry_picking_files/summary_CP_VUSDUALXP_batch5.csv', index_col=0)[cp_col]

pdest_layout_df = pd.concat([pdest_layout_b5_df, pdest_layout_initial_batches_df])

# Add metadata
meta_cols = ['orf_id', 'mut_id', 'aa_change', 'nt_change', 'symbol']
pdest_layout_df = pdest_layout_df.merge(allele_info_df[meta_cols], on=['orf_id', 'mut_id'], how='left')
pdest_layout_df[["aa_change", "nt_change"]] = pdest_layout_df[["aa_change", "nt_change"]].fillna("WT")
pdest_layout_df["orf_id"] = pdest_layout_df["orf_id"].astype("Int64").astype(str)
pdest_layout_df["orf_id"] = pdest_layout_df["orf_id"].replace("<NA>", np.nan)

# Replace missing symbols
pdest_layout_df["symbol"] = pdest_layout_df["symbol"].fillna(pdest_layout_df["orf_id"].map(orf_to_symbol))

In [117]:
pdest_layout_df.to_csv(f'{dualipa_outputs}/dualipa_experimental_layout.csv', index=None)